In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets
import numpy as np
import matplotlib.pyplot as plt

from ecc import *

# Module 5: ECDSA — Digital Signatures

**Where we are:** In Module 4 we saw ElGamal encryption — Bob masks a message
with a shared secret derived from Alice's public key, and Alice unmasks it with
her private key. ECDSA is the same family of ideas, but instead of *encrypting*
a message, we're *proving we authored it* — without revealing the private key.

ElGamal had a signature scheme too (not just encryption). DSA refined it, and
ECDSA transplanted it onto elliptic curves — the same move Koblitz and Miller
made for key exchange. The algebraic structure is identical, just performed in
the harder group.

## The one-way function fires twice

Every Bitcoin transaction uses the one-way function $P = d \times G$ in two
different moments:

**Moment 1 — key generation** (happened once, when the wallet was created):
you picked a secret $d$, computed $P = d \times G$, published $P$. Nobody can
reverse $P$ back to $d$. That's the alias.

**Moment 2 — signing** (happening right now, for this transaction): you pick a
fresh random $k$, compute $R = k \times G$. The one-way function fires *again*.
$R$ is public, but $k$ is hidden inside it. You then weave $k$, your private
key $d$, and the message hash together into a single number $s$. The signature
is $(r, s)$ where $r$ is the x-coordinate of $R$.

If anyone could reverse $R$ back to $k$, they could solve for $d$ from the
signing equation. Both moments rely on the same hardness: given a point and
the generator, you cannot find the scalar.

## 5.1 Signing — what the signer actually does

### The concrete situation

Alice has 0.5 BTC sitting in an address. That address is derived from her
public key $P$, which is derived from her private key $d$. She wants to send
0.3 BTC to Bob.

She creates a transaction: "Move 0.3 BTC from address(P) to address(Bob)."
This transaction is going to be broadcast to the entire network — everyone
will see it. The network needs to verify: **does the person who created this
transaction actually control the funds at address(P)?**

Alice can't show her private key $d$ — the whole network would see it and
steal her remaining 0.2 BTC. Instead, she produces a **signature**: a pair
of numbers $(r, s)$ that proves she knows $d$ without revealing it. The
network checks the signature against her public key $P$, confirms it's valid,
and the transaction goes through.

Here's what happens step by step:

```
  Alice knows: private key d, public key P = d×G, message to sign

  Step 1: Hash the message
           z = SHA-256(message)
           This compresses the message into a fixed-size number.

  Step 2: Pick a random nonce k
           Fresh, secret, never reused. (We'll see why in 5.4.)

  Step 3: Compute R = k × G
           The one-way function fires. R is a curve point.
           r = R's x-coordinate mod N
           This r becomes half the signature — it's the "commitment"
           to the random nonce, without revealing k.

  Step 4: Compute s = k⁻¹ · (z + r·d) mod N
           This is the core equation. It ties together:
           - z (the message — proves WHAT was signed)
           - r (the nonce commitment — proves THIS signing session)
           - d (the private key — proves WHO signed)
           And k⁻¹ scrambles it all so d can't be extracted.

  Step 5: Output signature (r, s)
           Two numbers. That's it. No private key revealed.
```

**Why does the equation work as a signature?** Because $s$ contains $d$ baked
in, but protected by $k$. The verifier can check that $s$ is consistent with
$P = dG$ (which carries $d$ inside it) without ever learning $d$ or $k$
individually. The private key "cancels out" during verification — just like
the shared secret cancelled out in ElGamal decryption.

## 5.2 Verification — what the verifier actually does

Alice broadcasts her transaction to the network: "Move 0.3 BTC from
address(P) to address(Bob)," along with her signature $(r, s)$ and her
public key $P$.

Every node that receives this transaction needs to answer one question:
**did the owner of this public key actually authorize this transaction?**
The node has $P$, the transaction data, and $(r, s)$. It does NOT know
$d$ or $k$. Can it check the signature is real?

```
  Verifier knows: public key P, message, signature (r, s)

  Step 1: Hash the message
           z = SHA-256(message)
           Same hash as the signer computed.

  Step 2: Compute s⁻¹ mod N
           This is the multiplicative inverse — the easy one,
           NOT the discrete log. Just Fermat's little theorem.

  Step 3: Compute two "weights"
           u₁ = z · s⁻¹ mod N    (how much of G)
           u₂ = r · s⁻¹ mod N    (how much of P)

  Step 4: Compute R' = u₁ × G + u₂ × P
           This is the key step. The verifier reconstructs a point
           using ONLY public information (G, P, z, r, s).
           If the signature is valid, R' should equal the signer's R.

  Step 5: Check: does R'_x mod N equal r?
           If yes → signature is valid. Alice has the private key.
           If no  → signature is forged or corrupted.
```

**The magic:** The verifier never knew $k$ or $d$, but by combining $G$ (which
everyone knows) and $P$ (which contains $d$ hidden inside), weighted by pieces
of the signature, he reconstructs the exact same point $R$ that the signer
computed from $k \times G$. The private key $d$ is hidden inside $P$, and it
"comes out" just enough to make the equation balance — but never enough to
be extracted.

In [ ]:
import hashlib

def ecdsa_sign(message: bytes, private_key: int) -> tuple:
    """Generate ECDSA signature (r, s)."""
    z = int.from_bytes(hashlib.sha256(message).digest(), 'big')
    
    while True:
        k = secrets.randbelow(SECP_N - 1) + 1
        R = scalar_mult(k, G)
        r = R.x % SECP_N
        if r == 0:
            continue
        
        k_inv = pow(k, SECP_N - 2, SECP_N)
        s = (k_inv * (z + r * private_key)) % SECP_N
        if s == 0:
            continue
        
        return (r, s)

def ecdsa_verify(message: bytes, signature: tuple, public_key: Point) -> bool:
    """Verify ECDSA signature."""
    r, s = signature
    z = int.from_bytes(hashlib.sha256(message).digest(), 'big')
    
    s_inv = pow(s, SECP_N - 2, SECP_N)
    u1 = (z * s_inv) % SECP_N
    u2 = (r * s_inv) % SECP_N
    
    R_prime = point_add(scalar_mult(u1, G), scalar_mult(u2, public_key))
    
    return R_prime.x % SECP_N == r

# Generate key pair
d = secrets.randbelow(SECP_N - 1) + 1
P = scalar_mult(d, G)

# Sign a message
msg = b"Hello, Bitcoin!"
sig = ecdsa_sign(msg, d)
r, s = sig

print("=== ECDSA Demonstration ===")
print(f"Message: {msg.decode()}")
print(f"Private key d: {hex(d)[:20]}...")
print(f"Public key P:  {P}")
print(f"\nSignature:")
print(f"  r = {hex(r)[:20]}...")
print(f"  s = {hex(s)[:20]}...")

# Verify
valid = ecdsa_verify(msg, sig, P)
print(f"\nVerification: {valid}  ✓")

# Tamper with message
tampered = b"Hello, Bitcorn!"
valid_tampered = ecdsa_verify(tampered, sig, P)
print(f"Tampered msg verification: {valid_tampered}  ✗ (correctly rejected)")

# Wrong key
wrong_key = scalar_mult(secrets.randbelow(SECP_N - 1) + 1, G)
valid_wrong = ecdsa_verify(msg, sig, wrong_key)
print(f"Wrong key verification: {valid_wrong}  ✗ (correctly rejected)")

## 5.3 Why Verification Works — The Proof

We said the private key "cancels out" during verification. Let's see it
algebraically — every step uses the group properties from Module 1.

The verifier computes $R' = u_1 \times G + u_2 \times P$. If the signature
is valid, $R'$ should equal the signer's $R = kG$. Let's prove it:

$$R' = u_1 G + u_2 P$$

Substitute $u_1 = z/s$, $u_2 = r/s$, and $P = dG$:

$$R' = \frac{z}{s} G + \frac{r}{s}(dG)$$

Factor out $G$ and $1/s$ (this uses **commutativity** and **associativity**):

$$= \frac{z + rd}{s} G$$

From the signing equation: $s = k^{-1}(z + rd)$, so $(z + rd)/s = k$:

$$R' = k G = R \quad \checkmark$$

The $d$ was hidden inside $P = dG$. The verification equation pulled it out
just enough to cancel with the $d$ baked into $s$ — but never exposed $d$
itself. The verifier only ever touched $P$ (which contains $d$ locked behind
the discrete log) and the public numbers $z$, $r$, $s$.

## 5.4 The Nonce Catastrophe

Look at the signing equation again: $s = k^{-1}(z + r \cdot d) \bmod N$.

The private key $d$ is protected by the nonce $k$. If $k$ is random and secret,
there's no way to isolate $d$ from a single equation — two unknowns, one equation.

**But what if you reuse $k$?**

If the same nonce $k$ is used for two different messages, the attacker gets two
equations with the same $k$:

$$s_1 = k^{-1}(z_1 + r \cdot d) \qquad s_2 = k^{-1}(z_2 + r \cdot d)$$

Subtract them — $d$ cancels out:

$$s_1 - s_2 = k^{-1}(z_1 - z_2)$$

Now solve for $k$:

$$k = \frac{z_1 - z_2}{s_1 - s_2} \bmod N$$

Once $k$ is known, plug back into either equation and solve for $d$:

$$d = r^{-1}(s \cdot k - z) \bmod N$$

**Game over.** The private key is exposed. All funds controlled by that key
can be stolen.

### This actually happened

**Sony PS3 (2010):** Sony signed every PS3 firmware update with ECDSA. They
used the same $k$ for every signature — literally a constant. The hacker group
fail0verflow noticed two signatures had the same $r$ value (same $k$ means
same $R = kG$ means same $r$), applied the algebra above, and extracted Sony's
master signing key. They could now sign any code as "Sony." The entire PS3
security model collapsed from one reused nonce.

**Android Bitcoin wallets (2013):** A bug in Android's `SecureRandom`
implementation caused some Bitcoin wallet apps to reuse nonces. Attackers
monitored the blockchain, found pairs of transactions from the same address
with the same $r$ value, extracted the private keys, and stole the bitcoin.
This wasn't theoretical — real money was taken.

### The fix: deterministic nonces (RFC 6979)

Don't rely on random number generators. Instead, compute $k$ deterministically:

$$k = \text{HMAC-SHA256}(\text{private key}, \text{message hash})$$

Same key + same message → same $k$ → same signature (reproducible).
Different message → different $k$ → no reuse possible.
The nonce is derived from the secret key, so it's unpredictable to attackers.

The code below demonstrates the catastrophe — two signatures with the same $k$,
and the private key extracted in a few lines of algebra.

In [ ]:
# Demonstration: nonce reuse catastrophe

victim_priv = secrets.randbelow(SECP_N - 1) + 1
victim_pub = scalar_mult(victim_priv, G)

# Victim signs two messages with the SAME nonce (fatal mistake)
k_reused = secrets.randbelow(SECP_N - 1) + 1
R_k = scalar_mult(k_reused, G)
r_val = R_k.x % SECP_N
k_inv = pow(k_reused, SECP_N - 2, SECP_N)

msg1 = b"Transfer 1 BTC to Alice"
msg2 = b"Transfer 2 BTC to Bob"
z1 = int.from_bytes(hashlib.sha256(msg1).digest(), 'big')
z2 = int.from_bytes(hashlib.sha256(msg2).digest(), 'big')

s1 = (k_inv * (z1 + r_val * victim_priv)) % SECP_N
s2 = (k_inv * (z2 + r_val * victim_priv)) % SECP_N

print("=== Nonce Reuse Attack ===")
print(f"Attacker sees two signatures with same r:")
print(f"  sig1: r = {hex(r_val)[:16]}..., s1 = {hex(s1)[:16]}...")
print(f"  sig2: r = {hex(r_val)[:16]}..., s2 = {hex(s2)[:16]}...")
print(f"  Same r → same nonce k was used!")

# Attacker recovers k
k_recovered = ((z1 - z2) * pow(s1 - s2, SECP_N - 2, SECP_N)) % SECP_N
print(f"\nAttacker recovers k: {k_recovered == k_reused}")

# Attacker recovers private key
d_recovered = (pow(r_val, SECP_N - 2, SECP_N) * (s1 * k_recovered - z1)) % SECP_N
print(f"Attacker recovers private key: {d_recovered == victim_priv}")
print(f"\n⚠️  NEVER reuse a nonce. Bitcoin uses RFC 6979 (deterministic k).")

## 5.5 From Math to Bytes — DER Encoding

The ECDSA math produces $(r, s)$ as big integers. But Bitcoin transactions are
byte streams. How does the math become wire format?

### DER encoding

ECDSA signatures use **DER** (Distinguished Encoding Rules), a variable-length
ASN.1 format:

```
0x30 || total_len || 0x02 || r_len || r_bytes || 0x02 || s_len || s_bytes
```

Each integer ($r$ and $s$) is encoded as a variable-length byte string — if the
high bit is set, a `0x00` padding byte is prepended to keep it positive. This
makes ECDSA signatures **variable length** (typically 71–73 bytes), plus a
sighash flag byte appended at the end.

### Low-S normalization (BIP 62)

Bitcoin requires $s \leq N/2$. If $s > N/2$, replace it with $N - s$.

Why? Remember from Module 5.3 that verification checks $R'_x = r$. It turns
out both $(r, s)$ and $(r, N-s)$ pass that check — they're both valid
signatures. That means anyone watching the network could take your transaction,
flip $s$ to $N-s$, and rebroadcast it. The transaction would still be valid
but would have a **different hash**. This is **transaction malleability** — it
broke early Bitcoin software that tracked transactions by hash.

The low-S rule eliminates the ambiguity: only one of the two $s$ values is
accepted, so nobody can flip it.

The code below builds DER encoding from scratch and demonstrates low-S
normalization. In Module 6, we'll see how Schnorr eliminates both of these
problems entirely (fixed 64 bytes, no malleability by design), and where
signatures live inside actual Bitcoin transactions.

In [ ]:
def der_encode_integer(value: int) -> bytes:
    """Encode a positive integer in DER format (0x02 || length || bytes)."""
    b = value.to_bytes((value.bit_length() + 7) // 8, 'big')
    if b[0] & 0x80:
        b = b'\x00' + b
    return bytes([0x02, len(b)]) + b

def der_encode_signature(r: int, s: int) -> bytes:
    """DER-encode an ECDSA signature: 0x30 || total_len || r_der || s_der."""
    r_der = der_encode_integer(r)
    s_der = der_encode_integer(s)
    body = r_der + s_der
    return bytes([0x30, len(body)]) + body

def low_s_normalize(s: int, N: int) -> int:
    """BIP 62: if s > N/2, replace with N - s."""
    if s > N // 2:
        return N - s
    return s

# Use the signature we generated in section 5.2
# r and s are the two components of any ECDSA signature
r_example = 0xDEADBEEF_CAFEBABE_12345678_9ABCDEF0_DEADBEEF_CAFEBABE_12345678_9ABCDEF0
s_example = 0xFEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210

print("=== DER Encoding (ECDSA) ===\n")
der_sig = der_encode_signature(r_example, s_example)
print(f"r = {hex(r_example)[:20]}...")
print(f"s = {hex(s_example)[:20]}...")
print(f"\nDER-encoded: {der_sig.hex()}")
print(f"DER length:  {len(der_sig)} bytes (variable)")
print(f"\nByte breakdown:")
print(f"  0x30       = SEQUENCE tag")
print(f"  0x{der_sig[1]:02x}       = total body length ({der_sig[1]} bytes)")
print(f"  0x02       = INTEGER tag (r)")
print(f"  0x{der_sig[3]:02x}       = r length ({der_sig[3]} bytes)")
print(f"  {der_sig[4:4+der_sig[3]].hex()[:40]}... = r value")
r_end = 4 + der_sig[3]
print(f"  0x02       = INTEGER tag (s)")
print(f"  0x{der_sig[r_end+1]:02x}       = s length ({der_sig[r_end+1]} bytes)")
print(f"  {der_sig[r_end+2:].hex()[:40]}... = s value")

# With sighash byte (SIGHASH_ALL = 0x01)
der_with_sighash = der_sig + bytes([0x01])
print(f"\n+ sighash byte (0x01) = {len(der_with_sighash)} bytes total on the wire")

# Low-S normalization
print(f"\n=== Low-S Normalization (BIP 62) ===\n")
s_high = SECP_N - 1
s_low = low_s_normalize(s_high, SECP_N)
print(f"Original s:   {hex(s_high)[:20]}... (> N/2)")
print(f"Normalized s: {hex(s_low)[:20]}... (= N - s)")
print(f"s > N/2: {s_high > SECP_N // 2} → replaced to prevent malleability")

print(f"\nECDSA DER signature: {len(der_with_sighash)} bytes on the wire (variable)")
print(f"In Module 6 we'll see Schnorr: fixed 64 bytes, no DER needed.")